# Heart Disease Risk Factors in the United States

### Exploratory Data Analysis (EDA)

## Project Overview

This project presents an exploratory data analysis (EDA) of the **Key Indicators of Heart Disease** dataset, based on the 2022 CDC Behavioral Risk Factor Surveillance System (BRFSS).

The objective of this analysis is to explore how demographic, lifestyle, and geographic factors are associated with heart attack prevalence in the United States.

The project focuses on identifying patterns and relationships in the data through visualization and descriptive analysis. It is intended to generate insights and hypotheses rather than establish causal relationships.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.express as px

In [ ]:
df = pd.read_csv("heart_2022_no_nans.csv")

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

## 2. Dataset

The analysis was performed using the **Key Indicators of Heart Disease** dataset, based on the 2022 CDC Behavioral Risk Factor Surveillance System (BRFSS).

For this project, the cleaned version of the dataset (without missing values) was used, following the course guidelines in order to focus on exploratory data analysis.

The final dataset contains **246,022 observations** and **40 variables**.

| Feature | Value |
|---------|-------|
| Dataset | Key Indicators of Heart Disease |
| Source | CDC BRFSS 2022 (via Kaggle) |
| Records | 246,022 |
| Variables | 40 |
| Missing values | None (cleaned dataset) |

## 3. Data Preparation

The original dataset was available in two versions: one containing missing values and a cleaned version without missing values.

Following the course guidelines, the cleaned dataset was selected in order to focus on exploratory data analysis (EDA) rather than data cleaning techniques.

Additional preprocessing included creating derived variables and improving the readability of selected features for analysis and visualization.

## 4. Exploratory Data Analysis

In [ ]:
df["HadHeartAttack2"] = df["HadHeartAttack"].map({"Yes": 1, "No": 0})

In [ ]:
df["HadHeartAttack2"].value_counts()

### 4.1 Angina

In [ ]:
df[["HadHeartAttack","HadAngina"]].value_counts(normalize=True)

In [ ]:
ct = pd.crosstab(df["HadAngina"], df["HadHeartAttack"], normalize='index')

colors = ['lightblue', 'salmon']

ct.plot(kind='bar', stacked=True, color=colors)
plt.title("Heart Attack by Angina Status")
plt.xlabel("Had Angina")
plt.ylabel("Proportion")
plt.legend(title="Heart Attack", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.show()


Angina status is strongly correlated with heart attack prevalence; individuals with angina show a much higher likelihood of experiencing a heart attack.

### 4.2 Geographic Analysis

Before conducting the geographic analysis, the data was prepared to create a heat map showing the percentage of heart attacks by state across the United States.

In [ ]:
us_state_to_abbrev ={
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands": "VI",
}

In [ ]:
df["State_Abbrev"] = df.State.map(us_state_to_abbrev)

In [ ]:
df.groupby("State_Abbrev")["HadHeartAttack2"].mean().sort_values(ascending=False)

In [ ]:
State_HeartAttack = df.groupby("State_Abbrev")["HadHeartAttack2"].mean().reset_index()
State_HeartAttack

In [ ]:
State_HeartAttack["HeartAttackPercent"] = (State_HeartAttack["HadHeartAttack2"] * 100).round()
State_HeartAttack

In [ ]:
state_names = df[["State_Abbrev","State"]].drop_duplicates()
state_names

In [ ]:
State_HeartAttack = State_HeartAttack.merge(state_names, on= "State_Abbrev", how= "left")
State_HeartAttack.drop("HadHeartAttack2", axis=1, inplace = True)

In [ ]:
State_HeartAttack.rename(columns={"HeartAttackPercent":"StatePositivePercent"},inplace=True)

In [ ]:
 fig= px.choropleth(
       State_HeartAttack,
       locations="State_Abbrev",
       color="StatePositivePercent",
       locationmode="USA-states",
       scope="usa",
       color_continuous_scale= "Reds",
       labels= {"StatePositivePercent":"Heart Attack Rate (%)"},
       hover_data= {"State":True,
                    "StatePositivePercent":True,
                    "State_Abbrev": False},
       title= "Heart Attack Rate Per State"
)
fig.show()

Overall, it appears that southeastern states have a higher prevalence of heart disease. This may be related to socioeconomic factors or differences in lifestyle and dietary habits. Further research could examine the lifestyle patterns in these states compared with those in the West Coast states.

In [ ]:
state_means = df.groupby("State")["HadHeartAttack2"].mean().sort_values(ascending=True)


least_dangerous_states = state_means.head(5)


most_dangerous_states = state_means.tail(5)


combined_states = pd.concat([least_dangerous_states, most_dangerous_states])


plt.figure(figsize=(7,6))
combined_states.plot.barh()

plt.title('Top 5 Least Dangerous and Top 5 Most Dangerous States by Heart Attack Incidence')
plt.xlabel('Average Had Heart Attack Incidence')
plt.ylabel('State')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

To further explore geographic differences in heart attack risk, the data was analyzed by age group across the different states. This analysis aims to identify which age groups are most affected in each state.

In [ ]:
age_hrt_at_pct = df.groupby(["State_Abbrev","AgeCategory"])["HadHeartAttack2"].mean().reset_index(name="HeartAttackPercent")
age_hrt_at_pct['HeartAttackPercent'] = (age_hrt_at_pct.HeartAttackPercent * 100).round()
age_hrt_at_pct

In [ ]:
age_hrt_at_pct[(age_hrt_at_pct.AgeCategory == "Age 30 to 34") & (age_hrt_at_pct.HeartAttackPercent >0)].sort_values(by= "HeartAttackPercent")

In [ ]:
max_risk_per_state = age_hrt_at_pct.loc[age_hrt_at_pct.groupby("State_Abbrev")["HeartAttackPercent"].idxmax()]
max_risk_per_state = max_risk_per_state.merge(state_names, on= "State_Abbrev", how= "left")
max_risk_per_state.sort_values(by="AgeCategory")

In [ ]:
fig= px.bar(
    max_risk_per_state,
    x = "State",
    y = "HeartAttackPercent",
    color = "AgeCategory",
    text = "AgeCategory",
    title = "Highest risk Age Category Per State"
)
fig.update_traces(textposition = "outside")
fig.update_layout(xaxis_tickangle = -45)
fig.show()

What is the life expectancy in the different states? Could the higher risk among younger age groups be related to lower life expectancy in these states?

In [ ]:
positive_heart_attack = df[df.HadHeartAttack2==1]
positive_heart_attack

In [ ]:
df.AgeCategory.unique()

In [ ]:
young_ages = ['Age 45 to 49', 'Age 40 to 44','Age 35 to 39', 'Age 25 to 29', 'Age 30 to 34',
       'Age 18 to 24']

In [ ]:
young_positive = positive_heart_attack[positive_heart_attack['AgeCategory'].isin(young_ages)]

total_by_state = positive_heart_attack['State'].value_counts()
young_by_state = young_positive['State'].value_counts()

young_percent = (young_by_state / total_by_state * 100).sort_values(ascending=False)


In [ ]:
total_by_state = positive_heart_attack['State'].value_counts()
young_by_state = young_positive['State'].value_counts()

young_percent = (young_by_state / total_by_state * 100).sort_values(ascending=False)

top10 = young_percent.head(10)


In [ ]:
top10_states = top10.index.tolist()


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=top10.index, y=top10.values, palette='coolwarm')

for i, val in enumerate(top10.values):
    plt.text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=10)

plt.title('States with the highest proportion of heart disease cases among adults under 50')
plt.ylabel('Percentage of young cases out of all cases per state(%)')
plt.xlabel('State')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


There's a significantly higher risk of heart attacks for young people in certain areas, which is different from the average risk in the general population. This points to unique factors or situations affecting this younger age group.

Understanding unique underlying factors is crucial.

Focused health programs are needed for prevention and treatment in younger populations.

In [ ]:
young_positive_top10 = young_positive[young_positive['State'].isin(top10_states)]


In [ ]:
ethnicity_by_state = pd.crosstab(
    young_positive_top10['State'],
    young_positive_top10['RaceEthnicityCategory'],
    normalize='index'
) * 100


In [ ]:
ethnicity_by_state.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='tab20')
plt.title('Ethnic Distribution of Young Heart Disease Cases in Top 10 States')
plt.ylabel('Percentage (%)')
plt.xlabel('State')
plt.legend(title='Race/Ethnicity', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


The analysis reveals substantial differences in the ethnic composition of young heart disease cases across the states with the highest rates, suggesting that ethnicity may be a relevant factor for further investigation.

### 4.3 Age & Gender

In [ ]:
age_sex_grouped = df.groupby(["AgeCategory","Sex"])["HadHeartAttack2"].mean()*100
age_sex_risk = age_sex_grouped.unstack()
age_sex_risk.plot.line(figsize= (8,6), marker = 'o')
plt.title("Heart Attack Risk by Age and Gender")
plt.xlabel("Age Group")
plt.ylabel("Risk(Mean of Heart Attacks)")
plt.xticks(ticks= range(len(age_sex_risk.index)),labels=age_sex_risk.index ,rotation= 60)
plt.tight_layout()
plt.show()




There is a clear increase in the prevalence of heart attacks with increasing age.

The risk is significantly higher in men compared to women.

In [ ]:
df.groupby("Sex")["HadHeartAttack2"].mean()*100

In [ ]:
grouped = df.groupby(["State","Sex"])["HadHeartAttack2"].mean().unstack()
grouped.columns.name = None
grouped.plot.bar(color= ["skyblue","salmon"],figsize= (12,6)),
width = 0.7


plt.title("Heart Attack Rate by State and Gender")
plt.ylabel("Mean of Heart Attack")
plt.xlabel("State")
plt.xticks(rotation = 45, ha= "right")
plt.legend(title ="Gender")
plt.tight_layout()
plt.show()

Men have nearly twice the prevalence of heart attacks compared to women.

### 4.4 BMI

In [ ]:
df["BMI_Category"] = pd.cut(
    df.BMI,
    bins=[0, 18.5, 25, 30, 100],
    labels=["Underweight", "Healthy Weight", "Overweight", "Obesity"],

)
df.BMI_Category

In [ ]:
bmi_heart_attack = df.groupby("BMI_Category")["HadHeartAttack2"].mean()

bmi_heart_attack.plot(
    kind="barh",
    figsize=(8, 5),
    title="Heart Attack Rate by BMI Category"
)

plt.xlabel("Heart Attack Rate")
plt.ylabel("BMI Category")
plt.show()

### 4.4 Physical Activity

In [ ]:
df_percent = (
              df.groupby(["AgeCategory","PhysicalActivities"])["HadHeartAttack"].value_counts(normalize=True)
              .rename("Percentage")
              .reset_index()
)

df_percent = df_percent[df_percent["HadHeartAttack"]=="Yes"]

plt.figure(figsize=(12,6))
sns.barplot(
    data=df_percent,
    x="AgeCategory",
    y="Percentage",
    hue="PhysicalActivities",
    order=sorted(df["AgeCategory"].unique()),
    palette="Set1"
)

plt.title("Heart Disease Pecentage by Age and Physical Activity")
plt.xlabel("Age Category")
plt.ylabel("Heart Disease Rate (%)")
plt.xticks(rotation=45)
plt.legend(title = "Physical Activity")
plt.tight_layout()
plt.show()

Physical activity is associated with a reduced risk of heart attack across all age groups.

### 4.5 Smoking

In [ ]:
import matplotlib.pyplot as plt

category_colors = {
    'Never smoked': '#2B5C8F',
    'Former smoker': '#4682B4',
    'Current smoker - now smokes every day': '#E74C3C',
    'Current smoker - now smokes some days': '#0099CC'
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

status_counts = df['SmokerStatus'].value_counts()
colors_list = [category_colors.get(cat, '#95A5A6') for cat in status_counts.index]

wedges, texts, autotexts = axes[0].pie(
    status_counts,
    autopct='%1.1f%%',
    pctdistance=0.75,
    startangle=140,
    colors=colors_list,
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2)
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_weight('bold')

axes[0].legend(
    wedges,
    status_counts.index,
    title="Smoker Status",
    loc="upper center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=2,
    frameon=False
)
axes[0].set_title('Population Share by Smoker Status (%)', fontsize=13, fontweight='bold', pad=15)

heart_rates = (df.groupby("SmokerStatus")["HadHeartAttack2"].mean() * 100).reindex(status_counts.index)
bar_colors = [category_colors.get(cat, '#95A5A6') for cat in heart_rates.index]

x_labels = [label.replace(' - ', '\n') for label in heart_rates.index]

bars = axes[1].bar(
    x_labels,
    heart_rates.values,
    color=bar_colors,
    width=0.55,
    edgecolor='none'
)

axes[1].set_title('Heart Attack Rate by Smoker Status (%)', fontsize=13, fontweight='bold', pad=15)
axes[1].set_ylabel('Heart Attack Prevalence (%)', fontsize=11)
axes[1].grid(axis='y', linestyle='--', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    axes[1].annotate(
        f'{height:.1f}%',
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

plt.tight_layout()
plt.show()

Smoking groups (both 'former smoker' and 'current smoker') exhibit higher rates of heart attack, while the 'never smoked' group presents the lowest rate.

While never-smokers represent the largest group in the general population, they constitute the smallest group among heart attack patients, highlighting a strong correlation between smoking and increased heart attack prevalence.

E-Cigarette

In [ ]:
import matplotlib.pyplot as plt

category_colors = {
    'Never used e-cigarettes in my entire life': '#2B5C8F',
    'Not at all (right now)': '#E74C3C',
    'Use them some days': '#4682B4',
    'Use them every day': '#0099CC'
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

ecig_counts = df['ECigaretteUsage'].value_counts()
colors_list = [category_colors.get(cat, '#8E44AD') for cat in ecig_counts.index]

wedges, texts, autotexts = axes[0].pie(
    ecig_counts,
    autopct='%1.1f%%',
    pctdistance=0.72,
    startangle=140,
    colors=colors_list,
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2)
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_weight('bold')

axes[0].legend(
    wedges,
    ecig_counts.index,
    title="E-Cigarette Usage",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    frameon=False
)
axes[0].set_title('Population Share by E-Cigarette Usage (%)', fontsize=13, fontweight='bold', pad=15)

heart_rates = (df.groupby("ECigaretteUsage")["HadHeartAttack2"].mean() * 100).reindex(ecig_counts.index)
bar_colors = [category_colors.get(cat, '#8E44AD') for cat in heart_rates.index]

x_labels = [label.replace(' ', '\n') for label in heart_rates.index]

bars = axes[1].bar(
    x_labels,
    heart_rates.values,
    color=bar_colors,
    width=0.55,
    edgecolor='none'
)

axes[1].set_xticks(range(len(x_labels)))
axes[1].set_xticklabels(x_labels, ha='center', fontsize=9)

axes[1].set_title('Heart Attack Rate by E-Cigarette Usage (%)', fontsize=13, fontweight='bold', pad=15)
axes[1].set_ylabel('Heart Attack Prevalence (%)', fontsize=11)
axes[1].grid(axis='y', linestyle='--', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    axes[1].annotate(
        f'{height:.1f}%',
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

plt.tight_layout()
plt.show()

E-cigarette usage shows a mixed impact on heart attack prevalence, with 'never used' and 'not at all (right now)' groups having higher rates than 'every day' users.

Further research is needed to understand why daily e-cigarette users exhibit the lowest heart attack prevalence in this dataset, as it contrasts with common assumptions.

In [ ]:
import matplotlib.pyplot as plt

ct = pd.crosstab(positive_heart_attack["ECigaretteUsage"], positive_heart_attack["SmokerStatus"], normalize="index")

custom_colors = ["#2B5C8F", "#4682B4", "#E74C3C", "#0099CC"]
colors_to_use = custom_colors[:len(ct.columns)]

ax = ct.plot(kind="bar", stacked=True, color=colors_to_use, figsize=(10, 4.5))

plt.title("Smoker Status by E-Cigarette Usage (Among those with Heart Attack)", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Proportion", fontsize=11)
plt.xlabel("", fontsize=11)

x_labels = [label.replace(' ', '\n') for label in ct.index]
plt.xticks(ticks=range(len(x_labels)), labels=x_labels, rotation=0, ha='center', fontsize=10)

plt.legend(title="Smoker Status", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)

plt.tight_layout()
plt.show()

Definitive conclusions on e-cigarette heart attack risk are challenging with this data. The shorter history of e-cigarettes and the significant presence of traditional smokers within the 'never used e-cigarettes' group obscure direct causal links, necessitating more controlled long-term studies.

##4.6 Alcohol

In [ ]:
df.groupby("AlcoholDrinkers")["HadHeartAttack2"].mean().plot.bar()

In [ ]:
Smoker_Drinker = df.groupby(["SmokerStatus", "AlcoholDrinkers"])["HadHeartAttack"].value_counts(normalize=True).unstack()
Smoker_Drinker["HeartAttackRate"] = (Smoker_Drinker["Yes"] * 100).round()
print(Smoker_Drinker)

In [ ]:
heatmap_data = Smoker_Drinker.HeartAttackRate.unstack()
heatmap_data

In [ ]:
plt.figure(figsize= (10, 6))
sns.heatmap(heatmap_data, annot=True,
            fmt=".0f",
            cmap="Reds",
            cbar_kws={"label":"Heart Attack Rate (%)"}
            )
plt.title("Heart Attack Rate by Smoking and Alcohol status")
plt.xlabel("AlcoholDrinkers")
plt.ylabel("SmokerStatus")
plt.tight_layout()
plt.show()

Based on this data, alcohol drinkers show a lower heart attack rate compared to non-drinkers. However, this observation requires further investigation to account for potential confounding factors and to determine causality.

## 5. Key Findings & Discussion

Based on the exploratory data analysis of the 2022 BRFSS dataset, several key patterns emerge:

1. **Geographic Concentration:** Heart disease risk is not uniformly distributed across the US. Southeastern states consistently show elevated risk levels, suggesting potential regional lifestyle, dietary, or healthcare access factors.
2. **Early-Onset Vulnerability:** While age remains a primary risk factor, specific geographic clusters demonstrate a concerningly high proportion of heart disease cases among adults under 50.
3. **Demographic Disparities:** The ethnic breakdown of young heart disease cases varies significantly by state, highlighting the importance of localized and population-specific health policy interventions.
4. **Angina as a Key Indicator:** Angina status shows a remarkably strong association with heart attack prevalence, validating its importance as a critical clinical red flag.


## 6. Limitations & Future Research

* **Data Limitations:** The dataset relies on self-reported survey data, which may be subject to recall bias. Additionally, as a cross-sectional study, causality cannot be established.
* **Future Directions:**
  * Incorporate socioeconomic indicators (e.g., median income, healthcare coverage per state) to test hypotheses regarding regional disparities.
  * Apply machine learning classification models (e.g., Logistic Regression, Random Forest) to predict heart attack risk based on lifestyle and demographic inputs.